## 10. Candidate Exogenous Feature Screening For SARIMAX

This table is a SARIMAX candidate pool, not final feature selection. EDA diagnostics such as CCF, Granger tests, VIF, missingness, and stability checks identify plausible regressors. The final SARIMAX specification must be selected later with rolling-origin or walk-forward forecasting performance. In short: EDA screens candidate regressors; out-of-sample rolling-origin forecasting selects the final SARIMAX specification.

In [12]:
coverage_lookup = coverage.set_index('variable')
stationarity_lookup = stationarity.set_index('variable')


def classify_candidate(row: pd.Series) -> tuple[str, str]:
    blockers = []
    warnings = []
    signals = []

    if not row.get('has_lag_feature', False):
        blockers.append('no lagged feature engineered in this notebook/src/features.py')
    elif pd.isna(row.get('feature_lag_quarters')):
        blockers.append('no candidate lag selected after availability/sample screen')
    elif row.get('forecast_origin_safe_under_assumption') is False:
        blockers.append('unsafe under assumed publication-lag metadata')

    if pd.notna(row.get('vif')) and row.get('vif') > VIF_THRESHOLD:
        warnings.append('high VIF in primary candidate matrix')
    if row.get('late_start_reduces_estimation_sample', False):
        warnings.append('starts later than full sample; reduces estimation window')
    if row.get('substantial_missingness_warning', False):
        warnings.append('substantial missingness warning')
    if row.get('candidate_feature') in row.get('oil_alternative_features', set()):
        warnings.append('alternative oil proxy; use at most one of WTI/Brent in a SARIMAX specification')

    if pd.notna(row.get('candidate_abs_corr')) and row.get('candidate_abs_corr') >= SIGNAL_CORR_THRESHOLD:
        signals.append('candidate-lag CCF signal')
    if pd.notna(row.get('candidate_granger_p_value')) and row.get('candidate_granger_p_value') <= SIGNAL_P_THRESHOLD:
        signals.append('candidate-lag Granger screening signal')

    if blockers:
        return 'exclude', '; '.join(blockers + warnings) if warnings else '; '.join(blockers)
    if not signals:
        return 'optional', '; '.join(warnings + ['weak candidate-lag CCF/Granger screening signal']) if warnings else 'weak candidate-lag CCF/Granger screening signal'
    strong_signal = (
        pd.notna(row.get('candidate_abs_corr')) and row.get('candidate_abs_corr') >= 0.40
    ) or (
        pd.notna(row.get('candidate_granger_p_value')) and row.get('candidate_granger_p_value') <= 0.05
    )
    if strong_signal and not row.get('substantial_missingness_warning', False) and not (pd.notna(row.get('vif')) and row.get('vif') > VIF_THRESHOLD):
        return 'strong candidate', '; '.join(warnings + signals) if warnings else '; '.join(signals)
    if warnings:
        return 'optional/shorter-sample candidate', '; '.join(warnings + signals)
    return 'candidate', '; '.join(signals)


def build_target_shortlist(
    target: str,
    candidate_artifacts: dict[str, object],
    best_predictive: pd.DataFrame,
    granger_frame: pd.DataFrame,
    message_prefix: str,
    show_result: bool = True,
) -> dict[str, object]:
    candidate_diag = candidate_artifacts['candidate_feature_diagnostics']
    candidate_diag_lookup = candidate_diag.set_index('candidate_feature') if not candidate_diag.empty else pd.DataFrame()
    chosen_feature_by_base = {lag_base_name(feature): feature for feature in candidate_artifacts['candidate_features']}
    vif_frame = candidate_artifacts['vif']
    vif_lookup = vif_frame.set_index('candidate_feature')['vif'].to_dict() if not vif_frame.empty else {}
    ccf_lookup = best_predictive.set_index('variable')
    granger_lookup_frame = granger_frame.set_index('variable')
    oil_alternative_features = set(candidate_artifacts['oil_pair_decision']['alternative_oil_feature']) if not candidate_artifacts['oil_pair_decision'].empty else set()

    shortlist_rows = []
    for variable in EXTERNAL_BASE_COLS:
        chosen_feature = chosen_feature_by_base.get(variable)
        availability_row = feature_availability.loc[feature_availability['candidate_feature'] == chosen_feature].iloc[0].to_dict() if chosen_feature in set(feature_availability['candidate_feature']) else {}
        candidate_row = candidate_diag_lookup.loc[chosen_feature].to_dict() if chosen_feature in candidate_diag_lookup.index else {}
        row = {
            'target': target,
            'variable': variable,
            'candidate_feature': chosen_feature,
            'has_lag_feature': variable in lagged_feature_bases,
            'integration_order': stationarity_lookup.loc[variable, 'integration_order'] if variable in stationarity_lookup.index else None,
            'best_ccf_lag': int(ccf_lookup.loc[variable, 'lag_quarters']) if variable in ccf_lookup.index else np.nan,
            'best_ccf_corr': ccf_lookup.loc[variable, 'correlation'] if variable in ccf_lookup.index else np.nan,
            'candidate_lag': candidate_row.get('candidate_lag', np.nan),
            'candidate_corr': candidate_row.get('candidate_corr', np.nan),
            'candidate_abs_corr': candidate_row.get('candidate_abs_corr', np.nan),
            'candidate_granger_p_value': candidate_row.get('candidate_granger_p_value', np.nan),
            'min_granger_p_value': granger_lookup_frame.loc[variable, 'min_granger_p_value'] if variable in granger_lookup_frame.index else np.nan,
            'vif': vif_lookup.get(chosen_feature, np.nan),
            'assumed_min_safe_lag_quarters': availability_assumptions.set_index('variable').loc[variable, 'assumed_min_safe_lag_quarters'] if variable in set(availability_assumptions['variable']) else np.nan,
            'feature_lag_quarters': availability_row.get('feature_lag_quarters', np.nan),
            'forecast_origin_safe_under_assumption': availability_row.get('forecast_origin_safe_under_assumption', False),
            'available_observations': coverage_lookup.loc[variable, 'available_observations'] if variable in coverage_lookup.index else np.nan,
            'coverage_pct': coverage_lookup.loc[variable, 'coverage_pct'] if variable in coverage_lookup.index else np.nan,
            'missing_pct': coverage_lookup.loc[variable, 'missing_pct'] if variable in coverage_lookup.index else np.nan,
            'first_non_null': coverage_lookup.loc[variable, 'first_non_null'] if variable in coverage_lookup.index else None,
            'last_non_null': coverage_lookup.loc[variable, 'last_non_null'] if variable in coverage_lookup.index else None,
            'coverage_class': coverage_lookup.loc[variable, 'coverage_class'] if variable in coverage_lookup.index else None,
            'late_start_reduces_estimation_sample': bool(coverage_lookup.loc[variable, 'late_start_reduces_estimation_sample']) if variable in coverage_lookup.index else True,
            'substantial_missingness_warning': bool(coverage_lookup.loc[variable, 'substantial_missingness_warning']) if variable in coverage_lookup.index else True,
            'oil_alternative_features': oil_alternative_features,
        }
        row['screening_status'], row['reason'] = classify_candidate(pd.Series(row))
        row.pop('oil_alternative_features', None)
        shortlist_rows.append(row)

    shortlist = pd.DataFrame(shortlist_rows).sort_values(['screening_status', 'candidate_granger_p_value', 'candidate_abs_corr'], ascending=[True, True, False])
    strong_or_candidate_features = shortlist.loc[shortlist['screening_status'].isin(['strong candidate', 'candidate', 'optional/shorter-sample candidate']), 'candidate_feature'].dropna().tolist()
    if strong_or_candidate_features:
        message = message_prefix + ': ' + ', '.join(strong_or_candidate_features)
    else:
        message = f'No external variables passed the candidate screening checks for {target}. Revisit feature engineering before SARIMAX modelling.'

    if show_result:
        display(shortlist.round({'best_ccf_corr': 3, 'candidate_corr': 3, 'candidate_abs_corr': 3, 'candidate_granger_p_value': 4, 'min_granger_p_value': 4, 'vif': 2, 'missing_pct': 3, 'coverage_pct': 3}))
        display(Markdown(message + '\n\nThese are not final feature-selection results. The modelling stage should compare a small set of nested SARIMAX specifications with walk-forward RMSE/MAE against seasonal-naive and SARIMA baselines, not exhaustive subset search on this small sample.'))

    return {
        'shortlist': shortlist,
        'chosen_feature_by_base': chosen_feature_by_base,
        'message': message,
        'strong_or_candidate_features': strong_or_candidate_features,
    }


headline_shortlist_artifacts = build_target_shortlist(
    TARGET,
    headline_candidate_artifacts,
    best_predictive_ccf,
    granger,
    'SARIMAX candidate pool from headline CPI EDA screening',
    show_result=True,
)
shortlist = headline_shortlist_artifacts['shortlist']
chosen_feature_by_base = headline_shortlist_artifacts['chosen_feature_by_base']

cpi_family_vif_lookup = cpi_family_vif.set_index('candidate_feature')['vif'].to_dict() if not cpi_family_vif.empty else {}
cpi_family_sarimax_rows = []
headline_ccf_lookup = best_predictive_ccf.set_index('variable')
headline_granger_lookup_frame = granger.set_index('variable')
for variable in [column for column in CPI_DIAGNOSTIC_BASE_COLS if column in df_model.columns]:
    lag_feature = 'trimmed_mean_cpi_yoy_lag1' if variable == 'trimmed_mean_cpi_yoy' else None
    diagnostic_lag = feature_lag(lag_feature) if lag_feature else int(headline_ccf_lookup.loc[variable, 'lag_quarters']) if variable in headline_ccf_lookup.index else np.nan
    row = {
        'variable': variable,
        'lag_feature_reviewed': lag_feature,
        'integration_order': stationarity_lookup.loc[variable, 'integration_order'] if variable in stationarity_lookup.index else None,
        'best_ccf_lag': int(headline_ccf_lookup.loc[variable, 'lag_quarters']) if variable in headline_ccf_lookup.index else np.nan,
        'best_ccf_corr': headline_ccf_lookup.loc[variable, 'correlation'] if variable in headline_ccf_lookup.index else np.nan,
        'diagnostic_lag': diagnostic_lag,
        'diagnostic_lag_corr': ccf_corr_at_lag(variable, diagnostic_lag) if pd.notna(diagnostic_lag) else np.nan,
        'diagnostic_lag_granger_p_value': granger_p_at_lag(variable, diagnostic_lag) if pd.notna(diagnostic_lag) else np.nan,
        'min_granger_p_value': headline_granger_lookup_frame.loc[variable, 'min_granger_p_value'] if variable in headline_granger_lookup_frame.index else np.nan,
        'vif_if_lag_feature_reviewed': cpi_family_vif_lookup.get(lag_feature, np.nan),
        'screening_note': 'CPI-family diagnostic; keep out of generic external SARIMAX pool pending out-of-sample tests.',
    }
    cpi_family_sarimax_rows.append(row)

cpi_family_sarimax_diagnostics = pd.DataFrame(cpi_family_sarimax_rows)
display(Markdown('CPI-family diagnostics below are reviewed separately from ordinary exogenous macro indicators because they are alternative CPI measures or headline-minus-underlying inflation transforms.'))
display(cpi_family_sarimax_diagnostics.round({'best_ccf_corr': 3, 'diagnostic_lag_corr': 3, 'diagnostic_lag_granger_p_value': 4, 'min_granger_p_value': 4, 'vif_if_lag_feature_reviewed': 2}))

,target,variable,candidate_feature,has_lag_feature,integration_order,best_ccf_lag,best_ccf_corr,candidate_lag,candidate_corr,candidate_abs_corr,candidate_granger_p_value,min_granger_p_value,vif,assumed_min_safe_lag_quarters,feature_lag_quarters,forecast_origin_safe_under_assumption,available_observations,coverage_pct,missing_pct,first_non_null,last_non_null,coverage_class,late_start_reduces_estimation_sample,substantial_missingness_warning,screening_status,reason
0,cpi_yoy,unemployment_rate,unemployment_rate_lag4,True,I(1),3,-0.312,4.0,-0.226,0.226,0.1817,0.0088,7.50,1,4.0,True,124,1.000,0.000,1995Q1,2025Q4,core_long_sample_candidate,False,False,candidate,candidate-lag CCF signal
4,cpi_yoy,wage_price_index,None,False,I(2),2,0.065,NaN,NaN,NaN,NaN,0.7792,NaN,1,NaN,False,114,0.919,0.081,1997Q3,2025Q4,shorter_sample_candidate,True,False,exclude,no lagged feature engineered in this notebook/...
6,cpi_yoy,producer_price_index,None,False,I(1),2,0.555,NaN,NaN,NaN,NaN,0.0068,NaN,1,NaN,False,110,0.887,0.113,1998Q3,2025Q4,shorter_sample_candidate,True,False,exclude,no lagged feature engineered in this notebook/...
8,cpi_yoy,commodity_price_index,None,False,I(1),2,0.263,NaN,NaN,NaN,NaN,0.0230,NaN,1,NaN,False,124,1.000,0.000,1995Q1,2025Q4,core_long_sample_candidate,False,False,exclude,no lagged feature engineered in this notebook/...
10,cpi_yoy,wti_price,None,False,I(1),3,0.348,NaN,NaN,NaN,NaN,0.0000,NaN,0,NaN,False,102,0.823,0.177,2000Q3,2025Q4,shorter_sample_candidate,True,False,exclude,no lagged feature engineered in this notebook/...
12,cpi_yoy,brent_price,None,False,I(0),1,0.467,NaN,NaN,NaN,NaN,0.0084,NaN,0,NaN,False,74,0.597,0.403,2007Q3,2025Q4,limited_sample_candidate,True,True,exclude,no lagged feature engineered in this notebook/...
14,cpi_yoy,aud_usd,None,False,I(1),4,0.032,NaN,NaN,NaN,NaN,0.1987,NaN,0,NaN,False,64,0.516,0.484,2010Q1,2025Q4,limited_sample_candidate,True,True,exclude,no lagged feature engineered in this notebook/...
16,cpi_yoy,household_spending,None,False,I(1),3,0.482,NaN,NaN,NaN,NaN,0.0037,NaN,1,NaN,False,54,0.435,0.565,2012Q3,2025Q4,exclude_low_coverage,True,True,exclude,no lagged feature engineered in this notebook/...
17,cpi_yoy,household_spending_growth,None,True,I(0),3,0.494,NaN,NaN,NaN,NaN,0.0016,NaN,1,NaN,False,53,0.427,0.573,2012Q4,2025Q4,exclude_low_coverage,True,True,exclude,no candidate lag selected after availability/s...
15,cpi_yoy,aud_usd_change,aud_usd_change_lag1,True,I(0),4,0.029,1.0,-0.008,0.008,0.1972,0.1972,1.85,0,1.0,True,63,0.508,0.492,2010Q2,2025Q4,limited_sample_candidate,True,True,optional,starts later than full sample; reduces estimat...


SARIMAX candidate pool from headline CPI EDA screening: unemployment_rate_lag4, brent_growth_lag1, cash_rate_change_lag1, inflation_expectations_business_lag1, cash_rate_lag2, wti_growth_lag1, ppi_growth_lag2, unemployment_rate_change_lag1, commodity_growth_lag1

These are not final feature-selection results. The modelling stage should compare a small set of nested SARIMAX specifications with walk-forward RMSE/MAE against seasonal-naive and SARIMA baselines, not exhaustive subset search on this small sample.

CPI-family diagnostics below are reviewed separately from ordinary exogenous macro indicators because they are alternative CPI measures or headline-minus-underlying inflation transforms.

,variable,lag_feature_reviewed,integration_order,best_ccf_lag,best_ccf_corr,diagnostic_lag,diagnostic_lag_corr,diagnostic_lag_granger_p_value,min_granger_p_value,vif_if_lag_feature_reviewed,screening_note
0,cpi_qoq,None,I(0),2,0.670,2,0.670,0.0000,0.0000,NaN,CPI-family diagnostic; keep out of generic ext...
1,trimmed_mean_cpi_qoq,None,I(0),1,0.717,1,0.717,0.0075,0.0008,NaN,CPI-family diagnostic; keep out of generic ext...
2,trimmed_mean_cpi_yoy,trimmed_mean_cpi_yoy_lag1,I(1),2,0.477,1,0.440,0.0016,0.0016,3.43,CPI-family diagnostic; keep out of generic ext...
3,headline_trimmed_mean_yoy_gap,None,I(0),1,0.659,1,0.659,0.8141,0.0359,NaN,CPI-family diagnostic; keep out of generic ext...
